# One check

The reconciliation loop closes the difference between what should exist and
what does. The core of whole loop is just one **check** which itself consists of three steps:

| step | question | reads | writes |
|---|---|---|---|
| **observe** | is the thing intent implies actually in storage? | S3, at the addresses intent implies | `materialized_models`, `materialized_nd_runs` |
| **gap** | what should exist, minus what does | a snapshot | nothing, it is a pure function |
| **act** | close the difference | — | submits a job, or records why not |

**Results flow upstream, so work starts downstream-first.** A reach's model
needs its downstream neighbour's model *and* ND library (the max-q stage
transfer line shapes this reach's geometry). Terminal reaches have
no downstream, so a fresh network starts building at its outlets and everything
else waits its turn.

 **NOTE:** Containers must be running with DB schema populated for this notebook to work.

In [ ]:
import json
import time

import pandas as pd

from recon import activity, check, db, gap, identity, intent, jobs, observe, processing, queue, storage
from recon.config import settings
from recon.workers import LocalDockerRunner

pd.set_option("display.max_colwidth", 42)

print(f"database  {settings.postgres_host}:{settings.postgres_port}/{settings.postgres_db}")
print(f"storage   s3://{settings.artifacts_s3_bucket} via {settings.aws_endpoint_url}")
print(f"images    {settings.build_model_image}\n          {settings.run_nd_scenarios_lisflood_cpu_image}\n          {settings.run_nd_scenarios_lisflood_gpu_image}")

## 1. Load Database and Author Intent (Only Needed for First Time)

Nothing exists yet in the database, Let's seed the network with network and lake polygons
`scripts/seed.py` does that, and it is the same script a deployment would run.

Apart from loading network and writing lake polygons to storage, it also write **intent** (`desired_state` rows) in the database.
Without intent, loop would just sit idle on reaches.

In [ ]:
import sys
sys.path.insert(0, "../scripts")
import seed

seed.seed(seed.DEFAULT_NETWORK_GPKG, seed.DEFAULT_NHF_GPKG)

with db.connect() as conn:
    conn.execute("TRUNCATE materialized_models, materialized_nd_runs, "
                 "materialized_kwse_runs, reach_processing, reach_activity")

display(pd.DataFrame(db.table_counts()))

## 2. Database is the Queue

There is no queue data structure in process, the database is the queue.
This allow us to restart reconciler without losing anything.

In [ ]:
due = pd.DataFrame(queue.due_reaches())
print(f"{len(due)} reaches due")
due.head(10)

## 3. Intent, and the address it implies

Lets Pick two reaches: a **terminal** and a
**non-terminal** just above one. For each, the loop resolves intent
(`COALESCE(desired_state.x, desired_state_defaults.x)`), builds the identity
object the job would build, and hashes it.

A object's identity is a hash of the inputs that
define it. All of those live in the database, so the loop can compute where a model *must* be
before any job runs. Because of this system can have direct access to artifacts rather than search and guess.

Notice that no job has run, yet we know exactly where each
reach's model must appear in the bucket.

In [ ]:
rows = db.query("""
    SELECT rn.reach_id, rn.is_terminal FROM reach_network rn
    JOIN reach_network ds ON ds.reach_id = rn.reach_to_id
    WHERE ds.is_terminal LIMIT 1""")
upstream_id = rows[0]["reach_id"]
terminal_id = db.one("SELECT reach_to_id AS r FROM reach_network WHERE reach_id=%s", (upstream_id,))["r"]
print(f"terminal reach:     {terminal_id}")
print(f"non-terminal above: {upstream_id}\n")

wanted = intent.effective(terminal_id)
identity_obj, identity_hash = identity.model_identity(wanted)
print("identity object the job will build:")
print(json.dumps(identity_obj, indent=2))
print(f"\npredicted identity hash: {identity_hash}")
print(f"predicted address:       {storage.model_base_path(terminal_id)}/{identity_hash}_<domain>/")

## 4. Check - Part 1 Observe

Observe looks at that one address. Anything else surrounding that address is simply not looked at.

Nothing is there yet, so no proof row is written. A proof row is a row that will be 
added in `materialized_*` tables, registering that the desired intent is materialized at this time.

In [ ]:
print("observe:", observe.observe_reach(terminal_id))
print("proof rows:", db.query("SELECT * FROM materialized_models"))

## 5. Check - Part 2 Gap

A Snapshot is state of database at one particular time for reaches of interest.

The snapshot is one query: this reach's materialized record, its downstream neighbour's
materialized record, and whether a job is already in flight.

`gap.calculate` purely works out of this snapshot. There is no interaction with database and storage, so the same snapshot always gives the same
answer. All the rules for what work to be performed in what order lives in `gap.py`

In [ ]:
for rid, label in ((terminal_id, "terminal"), (upstream_id, "non-terminal")):
    snap = check.load_snapshot(rid)
    print(f"{label} {rid}:")
    print(f"   snapshot: model_ok={snap.model_ok} ds_model_ok={snap.ds_model_ok} ds_nd_ok={snap.ds_nd_ok}")
    print(f"   decision: {gap.calculate(snap)}\n")

## 6. Check - Part 3 Act

`run_check` performs the above two steps and acts on the decision from Gap. For the
terminal reach that we have it means submitting a real `build_model` container and returning
immediately.

The fact that work is running lives in the **database**
(`current_step`, with the container id as the handle), not in this notebook's
memory. We can kill the kernel now and nothing is lost.

Checking again while the job runs does not resubmit. Checking the non-terminal records who it waits for.

In [ ]:
from recon.workers import job_env

runner = LocalDockerRunner(
    network=settings.docker_network,
    env_vars=job_env(),
    platform=settings.docker_platform,
    volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [])

print("terminal:    ", check.run_check(terminal_id, runner))
print("check again: ", check.run_check(terminal_id, runner), "   <- no resubmit")
print("non-terminal:", check.run_check(upstream_id, runner))
print()
pd.DataFrame(processing.in_flight())

## 7. Poll Jobs Separately

Separately there is a **job status routine** that asks the execution system (SEPEX)
what became of the jobs the database says are in flight. It records nothing about what jobs produced
it only clears the marker and requests a check. Whether anything was *produced* is
storage's question, answered by the next observe in the check we just requested.

In [ ]:
deadline = time.time() + 900
while time.time() < deadline:
    outcomes = jobs.status_pass(runner)
    if not outcomes:
        print("nothing in flight")
        break
    print(f"{time.strftime('%H:%M:%S')}  {outcomes[0]['status']:<10} {outcomes[0]['action']}")
    if outcomes[0]["status"] in ("succeeded", "failed"):
        break
    time.sleep(15)

Run Section 2 cell again and notice that the job is complete but
Loop is not running so the check request we made above will not be picked up, yet

## 8. The Next Check (via Observe) will Notice Desired State is Materialized now

If we run check now, observe will look at the predicted address again, and now the manifest is there.
Before adopting it, the loop verifies it, the manifest must belong to this
reach, sit in the folder its hash names, carry exactly the identity fields the
loop knows, and its identity object must re-hash to the value it claims. Only
then is the proof row (materialization record) written and applied_revision updated.

The job computed its identity **independently, inside the
container** and landed on the hash we predicted in step 3.

In [ ]:
print(check.run_check(terminal_id, runner))
print()
row = db.one("SELECT * FROM materialized_models WHERE reach_id=%s", (terminal_id,))
print("proof row: ", row)
print(f"\npredicted {identity_hash} == adopted {row['identity_hash']}:", identity_hash == row["identity_hash"])